### Dane

In [1]:
import pandas as pd 
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

df = pd.read_csv("data/dataset_invoices.csv")

print(df.shape)
df.head()

(39158, 27)


,business_code,cust_number,name_customer,clear_date,buisness_year,doc_id,document_create_date,due_in_date,invoice_currency,total_open_amount,...,invoice_age,doc_date,avg_delay_customer,customer_late_rate,customer_max_delay,customer_invoice_count,amount_vs_customer_avg,invoice_month,payment_terms_enc,currency_enc
0,U013,140103335,PARAM corp,23.01.2019,2019,1991818786,26.12.2018,29.01.2019,USD,13760.55,...,30,2018-12-26,NaN,NaN,NaN,0.0,NaN,12,59,1
1,U013,100009932,SYSCO IN trust,28.02.2019,2019,1991819555,28.12.2018,29.01.2019,USD,28225.48,...,30,2018-12-28,NaN,NaN,NaN,0.0,NaN,12,59,1
2,U001,200769623,WAL-MAR systems,09.01.2019,2019,1928539494,29.12.2018,14.01.2019,USD,580.67,...,15,2018-12-29,NaN,NaN,NaN,0.0,NaN,12,35,1
3,U001,200769623,WAL-MAR systems,09.01.2019,2019,1928537852,29.12.2018,14.01.2019,USD,5433.68,...,15,2018-12-29,0.0,0.0,0.0,1.0,9.357604,12,35,1
4,U001,200865666,RESTAU in,15.01.2019,2019,1928539272,29.12.2018,15.01.2019,USD,2272.20,...,15,2018-12-29,NaN,NaN,NaN,0.0,NaN,12,21,1


In [3]:

date_cols = [
    "document_create_date",
    "due_in_date",
    "baseline_create_date"
]

for col in date_cols:
    df[col] = pd.to_datetime(
        df[col],
        dayfirst=True,
        errors="coerce"
    )

df = df[df["paid_late"].notna()].copy()
df["paid_late"] = df["paid_late"].astype(int)

print("Wymiary po czyszczeniu:", df.shape)

Wymiary po czyszczeniu: (39158, 28)


Usunięto pełne duplikaty oraz pozostawiono wyłącznie faktury zamknięte, dla których znana jest wartość zmiennej docelowej paid_late.

### Feature engineering

In [ ]:

df["payment_term_invalid"] = (
    df["days_to_due"] < 0
).astype(int)

df.loc[
    df["days_to_due"] < 0,
    "days_to_due"
] = np.nan

df["invoice_month"] = (
    df["document_create_date"].dt.month
)

df["invoice_dayofweek"] = (
    df["document_create_date"].dt.dayofweek
)

df["invoice_year"] = (
    df["document_create_date"].dt.year
)

df["log_total_open_amount"] = np.log1p(
    df["total_open_amount"]
)


Utworzono cechy opisujące długość terminu płatności oraz moment wystawienia faktury. Ujemne terminy oznaczono jako niepoprawne, a ich wartość zastąpiono brakiem, który zostanie uzupełniony w pipeline. Kwotę faktury poddano transformacji logarytmicznej, aby ograniczyć wpływ bardzo wysokich wartości.

In [186]:
categorical_features = [
    "business_code",
    "cust_payment_terms"
]

numeric_features = [
    "log_total_open_amount",
    "payment_term_days",
    "payment_term_invalid",
    "invoice_month",
    "invoice_dayofweek",
    "invoice_year"
]

features = categorical_features + numeric_features

W modelu pominięto bezpośredni identyfikator klienta, aby ograniczyć zapamiętywanie kontrahentów i lepiej ocenić zdolność modelu do uogólniania. Zmienne takie jak days_late i clear_date nie są używane, ponieważ byłyby znane dopiero po zapłacie. Pominięto także walutę, ponieważ jest ona silnie powiązana z kodem biznesowym.

### Podział danych

In [187]:
df = df.sort_values(
    "document_create_date"
).reset_index(drop=True)

cutoff_date = pd.Timestamp("2019-11-19")

train_df = df[
    df["document_create_date"] < cutoff_date
].copy()

test_df = df[
    df["document_create_date"] >= cutoff_date
].copy()

print(
    "Train:",
    train_df["document_create_date"].min(),
    "-",
    train_df["document_create_date"].max()
)

print(
    "Test:",
    test_df["document_create_date"].min(),
    "-",
    test_df["document_create_date"].max()
)

Train: 2018-12-26 00:00:00 - 2019-11-18 00:00:00
Test: 2019-11-19 00:00:00 - 2020-03-02 00:00:00


In [188]:
X_train = train_df[features].copy()
y_train = train_df["paid_late"].copy()

X_test = test_df[features].copy()
y_test = test_df["paid_late"].copy()

print("Liczba obserwacji treningowych:", len(X_train))
print("Liczba obserwacji testowych:", len(X_test))

print(
    "Udział opóźnionych w treningu:",
    y_train.mean().round(3)
)

print(
    "Udział opóźnionych w teście:",
    y_test.mean().round(3)
)

Liczba obserwacji treningowych: 31384
Liczba obserwacji testowych: 7774
Udział opóźnionych w treningu: 0.426
Udział opóźnionych w teście: 0.393


Dane podzielono chronologicznie, dzięki czemu model jest trenowany na starszych fakturach i oceniany na późniejszych. Taki podział lepiej odpowiada rzeczywistemu zastosowaniu i ogranicza ryzyko przecieku informacji z przyszłości.

### Naiwny klasyfikator

In [189]:
dummy = DummyClassifier(strategy="most_frequent")

dummy.fit(X_train, y_train)

dummy_pred = dummy.predict(X_test)
dummy_prob = dummy.predict_proba(X_test)[:, 1]

print("DUMMY BASELINE")
print(classification_report(
    y_test,
    dummy_pred,
    zero_division=0
))

print(
    "ROC-AUC:",
    roc_auc_score(y_test, dummy_prob)
)

print(
    "PR-AUC:",
    average_precision_score(y_test, dummy_prob)
)

DUMMY BASELINE
              precision    recall  f1-score   support

           0       0.61      1.00      0.76      4716
           1       0.00      0.00      0.00      3058

    accuracy                           0.61      7774
   macro avg       0.30      0.50      0.38      7774
weighted avg       0.37      0.61      0.46      7774

ROC-AUC: 0.5
PR-AUC: 0.3933624903524569


### Regresja logistyczna

In [190]:
numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_features
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
])


In [191]:
baseline_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])

In [192]:
baseline_model.fit(X_train, y_train)

y_pred = baseline_model.predict(X_test)
y_prob = baseline_model.predict_proba(X_test)[:, 1]

In [193]:
print("\nLOGISTIC REGRESSION BASELINE")

print(classification_report(
    y_test,
    y_pred,
    target_names=["on_time", "late"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print(
    "ROC-AUC:",
    roc_auc_score(y_test, y_prob)
)

print(
    "PR-AUC:",
    average_precision_score(y_test, y_prob)
)


LOGISTIC REGRESSION BASELINE
              precision    recall  f1-score   support

     on_time       0.77      0.57      0.65      4716
        late       0.53      0.74      0.61      3058

    accuracy                           0.64      7774
   macro avg       0.65      0.65      0.63      7774
weighted avg       0.67      0.64      0.64      7774

Confusion matrix:
[[2688 2028]
 [ 808 2250]]
ROC-AUC: 0.7223169070572826
PR-AUC: 0.6321993408984714
